In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   EXACT REPLICATION — Paper 22                                               ║
║   "Data augmentation and refinement for recommender system:                  ║
║    A semi-supervised approach using maximum margin matrix factorization"     ║
║   Shaikh, Kagita, Kumar, Pujari  (arXiv:2306.13050v3)                       ║
╚══════════════════════════════════════════════════════════════════════════════╝
This notebook implements the ST-MMMF algorithm from the paper
"""

# ═══════════════════════════════════════════════════════════════════════════════
# 0.  CONFIGURE THESE PATHS BEFORE RUNNING
# ═══════════════════════════════════════════════════════════════════════════════

# --- Classic MovieLens 100K (943 users, 1682 items) ---
# You uploaded ml-100k.zip manually via Kaggle → Add Data → Upload.
# Kaggle extracts the zip and places files at:
#   /kaggle/input/ml-100k/ml-100k/u.data
DATASET_100K_PATH = "/kaggle/input/datasets/aksh0322/movielens-100k/ml-100k/u.data"

# --- Classic MovieLens 1M (6040 users, 3952 items) ---
# Download from: https://files.grouplens.org/datasets/movielens/ml-1m.zip
# Upload manually via Kaggle → Add Data → Upload, then the path will be:
#   /kaggle/input/ml-1m/ml-1m/ratings.dat
DATASET_1M_PATH = "/kaggle/input/datasets/aksh0322/movielens-1m/ml-1m/ratings.dat"

# ─── Output / checkpoint directories (Kaggle standard) ───────────────────────
import os
WORKING_DIR    = "/kaggle/working/"
CHECKPOINT_DIR = os.path.join(WORKING_DIR, "checkpoints")
OUTPUT_DIR     = os.path.join(WORKING_DIR, "outputs")
PREV_CHECKPOINT_DIR = "/kaggle/input/datasets/aksh0322/rsproject-checkpoints/checkpoints"
for _d in [CHECKPOINT_DIR, OUTPUT_DIR]:
    os.makedirs(_d, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# 1.  IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════
import sys, time, pickle, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import torch
import torch.nn as nn
from torch.optim import Adam

from tqdm import tqdm

# Baselines implemented in pure NumPy — no scikit-surprise required

# ═══════════════════════════════════════════════════════════════════════════════
# 2.  GPU SETUP
# ═══════════════════════════════════════════════════════════════════════════════
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 65)
print(f"  Device : {device}")
if torch.cuda.is_available():
    print(f"  GPU    : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM   : "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("=" * 65, "\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 3.  PAPER HYPERPARAMETERS  (Section 5.1 — do NOT change)
# ═══════════════════════════════════════════════════════════════════════════════

# Latent dimension d  (not explicitly stated; 10 is standard for MMMF)
D = 10

# Regularisation λ candidate set  {10^(i/16)}, ∀i ∈ {1,5,9,...,37}   (paper §5.1)
LAMBDA_CANDIDATES = [10 ** (i / 16) for i in range(1, 40, 4)]
# ≈ [1.155, 2.154, 4.018, 7.499, 13.98, 26.08, 48.70, 90.88, 169.6, 316.4]

# τ₁ selected as 49.99 from the search space {5,10,...,45,49.99} → used as fraction
TAU1 = 49.99 / 100          # = 0.4999   (paper: "τ₁ < 0.5")

# τ₂ fixed = 10  → fraction
TAU2 = 10.0 / 100           # = 0.10    (paper: "τ₂ < τ₁ is fixed to 10")

# Sampling percentage s = 100%   (paper: "s = 100 corresponding to smallest MAE")
SAMPLING_PCT = 1.0

# Maximum ratings augmented per outer iteration   (paper Section 4.2)
MAX_AUG_PER_ITER = 5000

# Number of outer ST-MMMF iterations   (paper Figure 6 x-axis)
N_OUTER_ITERS = 50

# Train/test split ratio   (paper: "randomly selected 80% for training, 20% test")
TRAIN_RATIO = 0.80

# Number of independent runs to average   (paper: "averaged over three runs")
N_RUNS = 3

# Rating scale
R = 5

# MMMF gradient optimisation settings
MMMF_LR         = 0.005     # learning rate c
MMMF_MAX_EPOCHS = 300       # max gradient iterations t₁
MMMF_PATIENCE   = 20        # early-stop patience
MMMF_BATCH_SIZE = 8192      # mini-batch size for GPU efficiency

print("Paper hyperparameters confirmed:")
print(f"  d={D}, τ₁={TAU1}, τ₂={TAU2}, s={SAMPLING_PCT}")
print(f"  MAX_AUG={MAX_AUG_PER_ITER}, N_OUTER={N_OUTER_ITERS}, N_RUNS={N_RUNS}")
print(f"  λ candidates: {[f'{l:.3f}' for l in LAMBDA_CANDIDATES]}\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 4.  DATA LOADING & PREPROCESSING
# ═══════════════════════════════════════════════════════════════════════════════

def _reindex(df):
    """Remap user/item IDs to contiguous integers starting at 1."""
    u_map = {u: i + 1 for i, u in enumerate(sorted(df["userId"].unique()))}
    m_map = {m: i + 1 for i, m in enumerate(sorted(df["movieId"].unique()))}
    df = df.copy()
    df["userId"]  = df["userId"].map(u_map)
    df["movieId"] = df["movieId"].map(m_map)
    return df


def _filter_cold_users(df, min_ratings=20):
    """Remove users with < min_ratings  (paper: "users with less than 20 removed")."""
    counts = df.groupby("userId").size()
    keep   = counts[counts >= min_ratings].index
    return df[df["userId"].isin(keep)].copy()


def _print_stats(df, name):
    n_users = df["userId"].nunique()
    n_items = df["movieId"].nunique()
    n_rat   = len(df)
    sparse  = 1 - n_rat / (n_users * n_items)
    print(f"  [{name}] Users={n_users}, Movies={n_items}, "
          f"Ratings={n_rat}, Sparsity={sparse:.0%}")


def load_100k():
    """
    Load the CLASSIC MovieLens 100K dataset (u.data).
    Verified stats: 943 users, 1682 movies, 100,000 ratings, scale 1–5 integer.
    Matches paper Table 1 exactly.
    """
    path = DATASET_100K_PATH

    if not os.path.exists(path):
        # Auto-search fallback in case Kaggle placed it at a slightly different path
        print(f"  [WARN] Not found at {path} — searching /kaggle/input/ ...")
        for root, dirs, files in os.walk("/kaggle/input"):
            for f in files:
                if f == "u.data":
                    path = os.path.join(root, f)
                    print(f"  [FOUND] {path}")
                    break
            if os.path.exists(path):
                break

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"ml-100k/u.data not found.\n"
            f"Upload ml-100k.zip via Kaggle → Add Data → Upload.\n"
            f"Download from: https://files.grouplens.org/datasets/movielens/ml-100k.zip"
        )

    # Classic u.data format: tab-separated, no header
    # columns: userId \t movieId \t rating \t timestamp
    df = pd.read_csv(path, sep="\t", header=None,
                     names=["userId", "movieId", "rating", "timestamp"])
    df["rating"] = df["rating"].astype(int).clip(1, 5)
    df = df[["userId", "movieId", "rating"]]
    df = _filter_cold_users(df)
    df = _reindex(df)
    _print_stats(df, "100K")
    return df


def load_1m():
    """
    Load the CLASSIC MovieLens 1M dataset (ratings.dat).
    Verified stats: 6040 users, 3952 movies, 1,000,209 ratings, scale 1–5 integer.
    Matches paper Table 1 exactly.
    """
    path = DATASET_1M_PATH

    if not os.path.exists(path):
        print(f"  [WARN] Not found at {path} — searching /kaggle/input/ ...")
        for root, dirs, files in os.walk("/kaggle/input"):
            for f in files:
                if f == "ratings.dat":
                    path = os.path.join(root, f)
                    print(f"  [FOUND] {path}")
                    break
            if os.path.exists(path):
                break

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"ml-1m/ratings.dat not found.\n"
            f"Upload ml-1m.zip via Kaggle → Add Data → Upload.\n"
            f"Download from: https://files.grouplens.org/datasets/movielens/ml-1m.zip"
        )

    # Classic ratings.dat format: double-colon separated, no header
    # columns: userId :: movieId :: rating :: timestamp
    df = pd.read_csv(path, sep="::", engine="python", header=None,
                     names=["userId", "movieId", "rating", "timestamp"])
    df["rating"] = df["rating"].astype(int).clip(1, 5)
    df = df[["userId", "movieId", "rating"]]
    df = _filter_cold_users(df)
    df = _reindex(df)
    _print_stats(df, "1M")
    return df


def train_test_split_df(df, train_ratio=TRAIN_RATIO, seed=42):
    """Reproducible random split (paper: 80/20, "averaged over three runs"→seed varies)."""
    rng   = np.random.RandomState(seed)
    idx   = rng.permutation(len(df))
    n     = int(len(df) * train_ratio)
    train = df.iloc[idx[:n]].copy().reset_index(drop=True)
    test  = df.iloc[idx[n:]].copy().reset_index(drop=True)
    return train, test

# ═══════════════════════════════════════════════════════════════════════════════
# 5.  SMOOTH HINGE LOSS   (Equations 3 & 11)
# ═══════════════════════════════════════════════════════════════════════════════

def h(z: torch.Tensor) -> torch.Tensor:
    """
    Smooth hinge h(z)  [Equation 3]:
        0           if z ≥ 1
        ½(1-z)²     if 0 < z < 1
        ½ - z       if z ≤ 0
    """
    out = torch.zeros_like(z)
    m_mid  = (z > 0) & (z < 1)
    m_low  = (z <= 0)
    out[m_mid] = 0.5 * (1 - z[m_mid]) ** 2
    out[m_low] = 0.5 - z[m_low]
    return out


def h_prime(z: torch.Tensor) -> torch.Tensor:
    """
    h'(z)  [Equation 11]:
        0       if z ≥ 1
        z - 1   if 0 < z < 1
        -1      if z ≤ 0
    """
    out = torch.zeros_like(z)
    m_mid = (z > 0) & (z < 1)
    m_low = (z <= 0)
    out[m_mid] = z[m_mid] - 1
    out[m_low] = -1.0
    return out

# ═══════════════════════════════════════════════════════════════════════════════
# 6.  MMMF MODEL   (Section 3 — Equations 2, 4–10)
# ═══════════════════════════════════════════════════════════════════════════════

class MMMF(nn.Module):
    """
    Maximum Margin Matrix Factorization (Rennie & Srebro, 2005).
    Learns U ∈ R^{N×d}, V ∈ R^{M×d}, Θ ∈ R^{N×(R-1)}
    by minimising the all-threshold smooth hinge loss (Equation 4).
    """

    def __init__(self, n_users: int, n_items: int, d: int = D, R: int = R):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.d = d
        self.R = R

        # U ∈ R^{N×d}  (Equation 1)
        self.U = nn.Parameter(0.01 * torch.randn(n_users, d))
        # V ∈ R^{M×d}
        self.V = nn.Parameter(0.01 * torch.randn(n_items, d))
        # Θ = [θ_{i,r}] ∈ R^{N×(R-1)} — R-1 thresholds per user, initially spread
        init_theta = torch.linspace(-1.5, 1.5, R - 1).unsqueeze(0).repeat(n_users, 1)
        self.Theta = nn.Parameter(init_theta + 0.01 * torch.randn(n_users, R - 1))

    # ── loss over observed entries (Equation 4) ─────────────────────────────
    def loss(self, user_idx, item_idx, ratings, lam):
        """
        Equation 4:  J = Σ_{r=1}^{R-1} Σ_{(i,j)∈Ω} h( T^r_{ij} (θ_{i,r} − x_{ij}) )
                       + (λ/2)(‖U‖²_F + ‖V‖²_F)
        """
        Ui   = self.U[user_idx]                            # B × d
        Vj   = self.V[item_idx]                            # B × d
        xij  = (Ui * Vj).sum(dim=1)                        # B  (scalar predictions)
        tij  = self.Theta[user_idx]                        # B × (R-1)

        # T^r_{ij}  (defined after Eq. 4):  +1 if r ≥ y_{ij},  -1 if r < y_{ij}
        y = ratings.float().unsqueeze(1)                   # B × 1
        r = torch.arange(1, self.R, device=device).float().unsqueeze(0)  # 1 × (R-1)
        T = torch.where(r >= y, torch.ones_like(r), -torch.ones_like(r)) # B × (R-1)

        # h(T^r_{ij} · (θ_{i,r} − x_{ij}))
        xij_exp = xij.unsqueeze(1).expand_as(tij)
        arg     = T * (tij - xij_exp)                      # B × (R-1)
        hinge   = h(arg).sum()

        reg = (lam / 2.0) * (self.U.norm() ** 2 + self.V.norm() ** 2)
        return hinge + reg

    # ── real-valued predictions X = U V^T ────────────────────────────────────
    @torch.no_grad()
    def predict_real(self) -> torch.Tensor:
        """X ← UV^T   (Equation 1)"""
        return self.U @ self.V.t()                         # N × M

    # ── discretise X into {1,...,R} using thresholds ─────────────────────────
    @torch.no_grad()
    def discretize(self, X=None) -> torch.Tensor:
        """
        ŷ_{ij} = r  if θ_{i,r-1} ≤ x_{ij} < θ_{i,r}
        (with θ_{i,0} = −∞, θ_{i,R} = +∞)
        """
        if X is None:
            X = self.predict_real()
        N = self.n_users
        Yhat = torch.full((N, self.n_items), self.R,
                          dtype=torch.long, device=device)
        for r in range(1, self.R + 1):
            lo = (self.Theta[:, r - 2].unsqueeze(1)
                  if r > 1
                  else torch.full((N, 1), float("-inf"), device=device))
            hi = (self.Theta[:, r - 1].unsqueeze(1)
                  if r < self.R
                  else torch.full((N, 1), float("+inf"), device=device))
            mask = (X >= lo) & (X < hi)
            Yhat[mask] = r
        return Yhat

    # ── ϑ_i = average gap between consecutive thresholds  (Section 4.1) ──────
    @torch.no_grad()
    def avg_gap(self) -> torch.Tensor:
        """
        ϑ_i = (1/(R-2)) Σ_{r=2}^{R-1} (θ_{i,r} − θ_{i,r-1})
        (Equation 12 text: "average gap between consecutive pairs of threshold values")
        If R=5, we have 4 thresholds → 3 consecutive gaps.
        """
        if self.R <= 2:
            return torch.ones(self.n_users, device=device)
        diffs = (self.Theta[:, 1:] - self.Theta[:, :-1]).clamp(min=1e-6)  # N × (R-2)
        return diffs.mean(dim=1)                           # N


# ═══════════════════════════════════════════════════════════════════════════════
# 7.  MMMF TRAINING
# ═══════════════════════════════════════════════════════════════════════════════

def train_mmmf(model: MMMF, train_df: pd.DataFrame, lam: float,
               desc: str = "MMMF") -> float:
    """
    Train MMMF with mini-batch Adam + early stopping.
    Enforces threshold ordering after each update (soft projection).
    Returns final training loss.
    """
    users = torch.tensor(train_df["userId"].values - 1,
                         dtype=torch.long, device=device)
    items = torch.tensor(train_df["movieId"].values - 1,
                         dtype=torch.long, device=device)
    rates = torch.tensor(train_df["rating"].values,
                         dtype=torch.float32, device=device)

    model.to(device).train()
    opt = Adam(model.parameters(), lr=MMMF_LR)

    n          = len(users)
    best_loss  = float("inf")
    patience_c = 0
    last_loss  = float("inf")

    pbar = tqdm(range(MMMF_MAX_EPOCHS), desc=f"    {desc}", leave=False, ncols=88)
    for epoch in pbar:
        perm       = torch.randperm(n, device=device)
        epoch_loss = 0.0

        for start in range(0, n, MMMF_BATCH_SIZE):
            idx = perm[start: start + MMMF_BATCH_SIZE]
            opt.zero_grad()
            loss = model.loss(users[idx], items[idx], rates[idx], lam)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            opt.step()

            # Enforce θ_{i,1} ≤ θ_{i,2} ≤ … ≤ θ_{i,R-1}  (Section 3.1)
            with torch.no_grad():
                model.Theta.data = torch.sort(model.Theta.data, dim=1).values

            epoch_loss += loss.item() * len(idx)

        avg_loss = epoch_loss / n
        pbar.set_postfix({"loss": f"{avg_loss:.4f}"})

        if avg_loss < best_loss - 1e-5:
            best_loss  = avg_loss
            patience_c = 0
        else:
            patience_c += 1
            if patience_c >= MMMF_PATIENCE:
                break

        last_loss = avg_loss

    return last_loss


# ═══════════════════════════════════════════════════════════════════════════════
# 8.  λ SELECTION  (paper: "regularisation value λ is tuned from the candidate
#                           set {10^(i/16)}, ∀i ∈ {1, 5, ..., 40}")
# ═══════════════════════════════════════════════════════════════════════════════

def select_lambda(train_df: pd.DataFrame, n_users: int, n_items: int,
                  val_ratio: float = 0.1) -> float:
    """
    Hold-out validation to pick the best λ from LAMBDA_CANDIDATES.
    Uses 10% of training data as an internal validation split.
    """
    print("  [λ-search] Evaluating candidates:", LAMBDA_CANDIDATES)
    rng = np.random.RandomState(0)
    idx = rng.permutation(len(train_df))
    n_val  = int(len(train_df) * val_ratio)
    val_df = train_df.iloc[idx[:n_val]].copy().reset_index(drop=True)
    tr_df  = train_df.iloc[idx[n_val:]].copy().reset_index(drop=True)

    best_lam  = LAMBDA_CANDIDATES[0]
    best_mae  = float("inf")
    for lam in tqdm(LAMBDA_CANDIDATES, desc="  λ-search", ncols=70):
        model = MMMF(n_users, n_items).to(device)
        train_mmmf(model, tr_df, lam, desc=f"λ={lam:.2f}")
        mae, _ = evaluate_mmmf(model, val_df)
        print(f"    λ={lam:.3f}  →  val-MAE={mae:.4f}")
        if mae < best_mae:
            best_mae = mae
            best_lam = lam

    print(f"  [λ-search] Best λ={best_lam:.3f}  (val-MAE={best_mae:.4f})\n")
    return best_lam


# ═══════════════════════════════════════════════════════════════════════════════
# 9.  EVALUATION
# ═══════════════════════════════════════════════════════════════════════════════

def evaluate_mmmf(model: MMMF, test_df: pd.DataFrame):
    """MAE and RMSE of discretised MMMF predictions on test_df."""
    model.eval()
    with torch.no_grad():
        Yhat = model.discretize()

    u = torch.tensor(test_df["userId"].values - 1, dtype=torch.long)
    j = torch.tensor(test_df["movieId"].values - 1, dtype=torch.long)
    y_pred = Yhat[u, j].cpu().numpy().astype(float)
    y_true = test_df["rating"].values.astype(float)

    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return float(mae), float(rmse)


def confusion_and_hr(y_true: np.ndarray, y_pred: np.ndarray, R: int = 5):
    """
    Confusion matrix and HR@K  (Section 5.2).
    HR@K = fraction of predictions with |predicted − actual| = K.
    """
    y_true = np.clip(np.round(y_true).astype(int), 1, R)
    y_pred = np.clip(np.round(y_pred).astype(int), 1, R)
    cm   = np.zeros((R, R), dtype=int)
    hr_k = np.zeros((R, R), dtype=float)
    for t, p in zip(y_true, y_pred):
        cm[t - 1, p - 1] += 1
    for r in range(1, R + 1):
        mask = (y_true == r)
        n_r  = mask.sum()
        if n_r == 0:
            continue
        preds_r = y_pred[mask]
        for k in range(R):
            hr_k[r - 1, k] = (np.abs(preds_r - r) == k).sum() / n_r
    return cm, hr_k


# ═══════════════════════════════════════════════════════════════════════════════
# 10. AUGMENTATION   (Equation 12 — high-confidence unobserved entries)
# ═══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def augmentation_candidates(model: MMMF, Y_mask: torch.Tensor, tau1: float):
    """
    Equation 12:  θ_{i,r-1} + ϑ_i·τ₁  <  x_{ij}  <  θ_{i,r} − ϑ_i·τ₁
    (θ_{i,0} = −∞, θ_{i,R} = +∞ by convention)

    Returns list of (user_0idx, item_0idx, rating_r).
    Only unobserved entries satisfying Eq. 12 are included.
    """
    X        = model.predict_real()        # N × M
    N, M     = X.shape
    vartheta = model.avg_gap()             # N  (average threshold gap per user)
    vt       = (vartheta * tau1).unsqueeze(1)  # N × 1

    candidates = []
    for r in range(1, model.R + 1):
        # Inner boundaries shifted by ϑ_i · τ₁
        if r == 1:
            lo_inner = torch.full((N, M), float("-inf"), device=device)
        else:
            lo_inner = (model.Theta[:, r - 2].unsqueeze(1) + vt).expand(N, M)

        if r == model.R:
            hi_inner = torch.full((N, M), float("+inf"), device=device)
        else:
            hi_inner = (model.Theta[:, r - 1].unsqueeze(1) - vt).expand(N, M)

        # Mask: inside high-confidence region AND unobserved
        in_region  = (X > lo_inner) & (X < hi_inner)
        high_conf  = in_region & (~Y_mask)

        ui, ji = high_conf.nonzero(as_tuple=True)
        for u, j in zip(ui.tolist(), ji.tolist()):
            candidates.append((u, j, r))

    return candidates


# ═══════════════════════════════════════════════════════════════════════════════
# 11. REFINEMENT   (Equation 13 — low-confidence observed entries)
# ═══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def refinement_candidates(model: MMMF, Y_mask: torch.Tensor, tau2: float):
    """
    Equation 13:  θ_{i,r} − ϑ_i·τ₂  <  x_{ij}  <  θ_{i,r} + ϑ_i·τ₂
    Returns set of (user_0idx, item_0idx) to remove from training.
    """
    X        = model.predict_real()
    N, M     = X.shape
    vartheta = model.avg_gap()
    vt       = (vartheta * tau2).unsqueeze(1)  # N × 1

    remove_set = set()
    for r in range(1, model.R):                  # thresholds r = 1 … R-1
        theta_r = model.Theta[:, r - 1].unsqueeze(1)  # N × 1
        lo = (theta_r - vt).expand(N, M)
        hi = (theta_r + vt).expand(N, M)
        near_boundary = (X > lo) & (X < hi) & Y_mask
        ui, ji = near_boundary.nonzero(as_tuple=True)
        for u, j in zip(ui.tolist(), ji.tolist()):
            remove_set.add((u, j))

    return remove_set


# ═══════════════════════════════════════════════════════════════════════════════
# 12. IMBALANCE HANDLING   (Section 4.2 — Equation 14)
# ═══════════════════════════════════════════════════════════════════════════════

def balance_and_sample(candidates, train_df: pd.DataFrame,
                       R: int = R, max_per_iter: int = MAX_AUG_PER_ITER):
    """
    Equation 14: quota for rating r  ∝  (1 − Z_r) / Σ_j (1 − Z_j)
    where Z_r = fraction of rating label r in current training set.
    Then Sample(Y^A, s) with s = SAMPLING_PCT  (here s=100% → take all, up to quota).
    """
    if not candidates:
        return []

    total  = max(len(train_df), 1)
    counts = train_df["rating"].value_counts()

    # Z_r for each rating level
    Z = {r: counts.get(r, 0) / total for r in range(1, R + 1)}

    # Equation 14: quota proportional to (1 - Z_r)
    denom  = sum(1 - Z[r] for r in range(1, R + 1))
    quota  = {
        r: int(max_per_iter * (1 - Z[r]) / max(denom, 1e-9))
        for r in range(1, R + 1)
    }

    # Group by rating
    by_r = {r: [] for r in range(1, R + 1)}
    for u, j, rating in candidates:
        by_r[rating].append((u, j, rating))

    sampled = []
    rng = np.random.default_rng()
    for r in range(1, R + 1):
        pool = by_r[r]
        k    = min(len(pool), quota[r])
        if k > 0:
            idx = rng.choice(len(pool), k, replace=False)
            sampled.extend(pool[i] for i in idx)

    return sampled


# ═══════════════════════════════════════════════════════════════════════════════
# 13. SURPRISE BASELINES
# ═══════════════════════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════════════════════
# 13. PURE-NUMPY BASELINE ALGORITHMS  (no scikit-surprise dependency)
#     All four match the paper's baselines exactly in algorithm family.
#     Implementations use SGD with early stopping, same as Surprise internals.
# ═══════════════════════════════════════════════════════════════════════════════

def _make_rating_matrix_sparse(train_df, n_users, n_items):
    """Return arrays of (user_indices, item_indices, ratings) — 0-indexed."""
    u = train_df["userId"].values - 1
    j = train_df["movieId"].values - 1
    r = train_df["rating"].values.astype(np.float32)
    return u, j, r


# ── SVD (Funk SVD — SGD with biases) ─────────────────────────────────────────
class _SVD:
    """
    Funk SVD: predict r_{ui} = μ + b_u + b_i + p_u · q_i
    Matches scikit-surprise SVD algorithm.
    """
    def __init__(self, n_factors=50, n_epochs=20, lr=0.005, reg=0.02,
                 rating_min=1, rating_max=5):
        self.k = n_factors; self.epochs = n_epochs
        self.lr = lr; self.reg = reg
        self.rmin = rating_min; self.rmax = rating_max

    def fit(self, train_df, n_users, n_items):
        rng = np.random.default_rng(42)
        k = self.k
        self.mu = train_df["rating"].mean()
        self.bu = np.zeros(n_users)
        self.bi = np.zeros(n_items)
        self.P  = rng.normal(0, 0.1, (n_users, k))
        self.Q  = rng.normal(0, 0.1, (n_items, k))
        u, j, r = _make_rating_matrix_sparse(train_df, n_users, n_items)
        for _ in range(self.epochs):
            idx = rng.permutation(len(u))
            for x in idx:
                ui, ji, ri = u[x], j[x], r[x]
                pred = (self.mu + self.bu[ui] + self.bi[ji]
                        + self.P[ui] @ self.Q[ji])
                e = ri - pred
                self.bu[ui] += self.lr * (e - self.reg * self.bu[ui])
                self.bi[ji] += self.lr * (e - self.reg * self.bi[ji])
                puf = self.P[ui].copy()
                self.P[ui] += self.lr * (e * self.Q[ji] - self.reg * self.P[ui])
                self.Q[ji] += self.lr * (e * puf         - self.reg * self.Q[ji])
        return self

    def predict(self, u_arr, j_arr):
        p = (self.mu + self.bu[u_arr] + self.bi[j_arr]
             + (self.P[u_arr] * self.Q[j_arr]).sum(axis=1))
        return np.clip(np.round(p), self.rmin, self.rmax)


# ── NMF (Non-negative Matrix Factorization — multiplicative updates) ──────────
class _NMF:
    """
    NMF with SGD updates (keeps factors non-negative).
    Matches scikit-surprise NMF algorithm.
    """
    def __init__(self, n_factors=15, n_epochs=50, reg_pu=0.06, reg_qi=0.06,
                 rating_min=1, rating_max=5):
        self.k = n_factors; self.epochs = n_epochs
        self.reg_pu = reg_pu; self.reg_qi = reg_qi
        self.rmin = rating_min; self.rmax = rating_max

    def fit(self, train_df, n_users, n_items):
        rng = np.random.default_rng(42)
        k = self.k
        self.mu = train_df["rating"].mean()
        self.bu = np.zeros(n_users)
        self.bi = np.zeros(n_items)
        self.P  = rng.uniform(0.1, 0.5, (n_users, k))
        self.Q  = rng.uniform(0.1, 0.5, (n_items, k))
        u, j, r = _make_rating_matrix_sparse(train_df, n_users, n_items)
        for _ in range(self.epochs):
            idx = rng.permutation(len(u))
            for x in idx:
                ui, ji, ri = u[x], j[x], r[x]
                pred = self.P[ui] @ self.Q[ji]
                if pred < 1e-9:
                    pred = 1e-9
                e = ri / pred
                # Multiplicative update (Surprise NMF style)
                self.P[ui] *= (e * self.Q[ji]) / (self.reg_pu + self.Q[ji])
                self.Q[ji] *= (e * self.P[ui]) / (self.reg_qi + self.P[ui])
                self.P[ui] = np.maximum(self.P[ui], 1e-9)
                self.Q[ji] = np.maximum(self.Q[ji], 1e-9)
        return self

    def predict(self, u_arr, j_arr):
        p = (self.P[u_arr] * self.Q[j_arr]).sum(axis=1)
        return np.clip(np.round(p), self.rmin, self.rmax)


# ── SVD++ (adds implicit feedback term y_i) ───────────────────────────────────
class _SVDpp:
    """
    SVD++: r_{ui} = μ + b_u + b_i + (p_u + |N(u)|^{-½} Σ_{j∈N(u)} y_j) · q_i
    Matches scikit-surprise SVD++ algorithm.
    """
    def __init__(self, n_factors=20, n_epochs=20, lr=0.007, reg=0.02,
                 rating_min=1, rating_max=5):
        self.k = n_factors; self.epochs = n_epochs
        self.lr = lr; self.reg = reg
        self.rmin = rating_min; self.rmax = rating_max

    def fit(self, train_df, n_users, n_items):
        rng = np.random.default_rng(42)
        k = self.k
        self.mu = train_df["rating"].mean()
        self.bu = np.zeros(n_users)
        self.bi = np.zeros(n_items)
        self.P  = rng.normal(0, 0.1, (n_users, k))
        self.Q  = rng.normal(0, 0.1, (n_items, k))
        self.Y  = rng.normal(0, 0.1, (n_items, k))  # implicit feedback vectors
        u, j, r = _make_rating_matrix_sparse(train_df, n_users, n_items)
        # Precompute implicit item sets per user
        from collections import defaultdict
        Nu = defaultdict(set)
        for ui, ji in zip(u, j):
            Nu[ui].add(ji)
        Nu = {ui: np.array(list(items)) for ui, items in Nu.items()}
        for _ in range(self.epochs):
            idx = rng.permutation(len(u))
            for x in idx:
                ui, ji, ri = u[x], j[x], r[x]
                nu_items = Nu.get(ui, np.array([], dtype=int))
                sqrt_nu = max(len(nu_items) ** 0.5, 1e-9)
                impl = self.Y[nu_items].sum(axis=0) / sqrt_nu if len(nu_items) else np.zeros(k)
                pu_impl = self.P[ui] + impl
                pred = (self.mu + self.bu[ui] + self.bi[ji]
                        + pu_impl @ self.Q[ji])
                e = ri - pred
                self.bu[ui] += self.lr * (e - self.reg * self.bu[ui])
                self.bi[ji] += self.lr * (e - self.reg * self.bi[ji])
                puf = self.P[ui].copy()
                self.P[ui] += self.lr * (e * self.Q[ji] - self.reg * self.P[ui])
                qjf = self.Q[ji].copy()
                self.Q[ji] += self.lr * (e * pu_impl    - self.reg * self.Q[ji])
                if len(nu_items):
                    self.Y[nu_items] += self.lr * (
                        (e / sqrt_nu) * qjf - self.reg * self.Y[nu_items])
        # Cache implicit sums for prediction
        self._Nu = Nu
        return self

    def predict(self, u_arr, j_arr):
        preds = []
        for ui, ji in zip(u_arr, j_arr):
            nu_items = self._Nu.get(ui, np.array([], dtype=int))
            sqrt_nu  = max(len(nu_items) ** 0.5, 1e-9)
            impl     = self.Y[nu_items].sum(axis=0) / sqrt_nu if len(nu_items) else np.zeros(self.k)
            p = (self.mu + self.bu[ui] + self.bi[ji]
                 + (self.P[ui] + impl) @ self.Q[ji])
            preds.append(np.clip(round(p), self.rmin, self.rmax))
        return np.array(preds)


# ── Co-Clustering ──────────────────────────────────────────────────────────────
class _CoClustering:
    """
    Gorakala & Usunier (2015) Co-Clustering.
    predict r_{ui} = avg_{cluster(u),cluster(i)} + avg_u - avg_{cluster(u)}
                   + avg_i - avg_{cluster(i)}
    Matches scikit-surprise CoClustering algorithm.
    """
    def __init__(self, n_cltr_u=3, n_cltr_i=3, n_epochs=20,
                 rating_min=1, rating_max=5):
        self.cu = n_cltr_u; self.ci = n_cltr_i
        self.epochs = n_epochs
        self.rmin = rating_min; self.rmax = rating_max

    def fit(self, train_df, n_users, n_items):
        rng = np.random.default_rng(42)
        u, j, r = _make_rating_matrix_sparse(train_df, n_users, n_items)
        # Initialise cluster assignments randomly
        clst_u = rng.integers(0, self.cu, size=n_users)
        clst_i = rng.integers(0, self.ci, size=n_items)

        self.mu_u = np.zeros(n_users)
        self.mu_i = np.zeros(n_items)
        for ui in range(n_users):
            mask = u == ui
            self.mu_u[ui] = r[mask].mean() if mask.any() else 0.0
        for ji in range(n_items):
            mask = j == ji
            self.mu_i[ji] = r[mask].mean() if mask.any() else 0.0

        for _ in range(self.epochs):
            # Compute co-cluster means
            coclust_sum   = np.zeros((self.cu, self.ci))
            coclust_count = np.zeros((self.cu, self.ci))
            for x in range(len(u)):
                cu_, ci_ = clst_u[u[x]], clst_i[j[x]]
                coclust_sum[cu_, ci_]   += r[x]
                coclust_count[cu_, ci_] += 1
            with np.errstate(invalid="ignore", divide="ignore"):
                coclust_mean = np.where(coclust_count > 0,
                                        coclust_sum / coclust_count, 0.0)

            clst_u_mean = np.array([
                coclust_mean[cu_, :].mean() for cu_ in range(self.cu)])
            clst_i_mean = np.array([
                coclust_mean[:, ci_].mean() for ci_ in range(self.ci)])

            # Reassign user clusters
            for ui in range(n_users):
                mask = u == ui
                if not mask.any():
                    continue
                costs = np.array([
                    np.sum((r[mask] - (
                        coclust_mean[cu_, clst_i[j[mask]]]
                        + self.mu_u[ui] - clst_u_mean[cu_]
                        + self.mu_i[j[mask]] - clst_i_mean[clst_i[j[mask]]]
                    ))**2) for cu_ in range(self.cu)])
                clst_u[ui] = int(np.argmin(costs))

            # Reassign item clusters
            for ji in range(n_items):
                mask = j == ji
                if not mask.any():
                    continue
                costs = np.array([
                    np.sum((r[mask] - (
                        coclust_mean[clst_u[u[mask]], ci_]
                        + self.mu_u[u[mask]] - clst_u_mean[clst_u[u[mask]]]
                        + self.mu_i[ji] - clst_i_mean[ci_]
                    ))**2) for ci_ in range(self.ci)])
                clst_i[ji] = int(np.argmin(costs))

        # Recompute final co-cluster means
        coclust_sum   = np.zeros((self.cu, self.ci))
        coclust_count = np.zeros((self.cu, self.ci))
        for x in range(len(u)):
            cu_, ci_ = clst_u[u[x]], clst_i[j[x]]
            coclust_sum[cu_, ci_]   += r[x]
            coclust_count[cu_, ci_] += 1
        with np.errstate(invalid="ignore", divide="ignore"):
            self.coclust_mean = np.where(coclust_count > 0,
                                         coclust_sum / coclust_count, 0.0)
        self.clst_u_mean = np.array([
            self.coclust_mean[cu_, :].mean() for cu_ in range(self.cu)])
        self.clst_i_mean = np.array([
            self.coclust_mean[:, ci_].mean() for ci_ in range(self.ci)])
        self.clst_u = clst_u
        self.clst_i = clst_i
        return self

    def predict(self, u_arr, j_arr):
        preds = []
        for ui, ji in zip(u_arr, j_arr):
            cu_ = self.clst_u[ui] if ui < len(self.clst_u) else 0
            ci_ = self.clst_i[ji] if ji < len(self.clst_i) else 0
            p = (self.coclust_mean[cu_, ci_]
                 + self.mu_u[ui] - self.clst_u_mean[cu_]
                 + self.mu_i[ji] - self.clst_i_mean[ci_])
            preds.append(np.clip(round(p), self.rmin, self.rmax))
        return np.array(preds)


# ── Best fixed hyperparameters (matches Surprise defaults, paper-compatible) ──
_BEST_PARAMS = {
    "SVD":           dict(n_factors=50,  n_epochs=20, lr=0.005, reg=0.02),
    "NMF":           dict(n_factors=15,  n_epochs=50),
    "SVD++":         dict(n_factors=20,  n_epochs=20, lr=0.007, reg=0.02),
    "Co-Clustering": dict(n_cltr_u=3,    n_cltr_i=3,  n_epochs=20),
}


def tune_baselines(train_df: pd.DataFrame, R: int = R):
    """Validate that all four baseline classes run without error."""
    print("  [Baselines] Verifying pure-NumPy baselines (no scikit-surprise) ...")
    n_users = train_df["userId"].max()
    n_items = train_df["movieId"].max()
    # Quick smoke-test on a tiny slice
    tiny = train_df.sample(min(200, len(train_df)), random_state=0)
    for name, cls, params in [
        ("SVD",           _SVD,           _BEST_PARAMS["SVD"]),
        ("NMF",           _NMF,           _BEST_PARAMS["NMF"]),
        ("SVD++",         _SVDpp,         _BEST_PARAMS["SVD++"]),
        ("Co-Clustering", _CoClustering,  _BEST_PARAMS["Co-Clustering"]),
    ]:
        algo = cls(**params, rating_min=1, rating_max=R)
        algo.fit(tiny, n_users, n_items)
        print(f"    {name}: OK")
    print("  [Baselines] All verified.\n")


def run_baseline(algo_name: str, train_df: pd.DataFrame,
                 test_df: pd.DataFrame, R: int = R):
    """Train a pure-NumPy baseline on train_df, evaluate MAE+RMSE on test_df."""
    n_users = max(train_df["userId"].max(), test_df["userId"].max())
    n_items = max(train_df["movieId"].max(), test_df["movieId"].max())

    cls_map = {
        "SVD":           _SVD,
        "NMF":           _NMF,
        "SVD++":         _SVDpp,
        "Co-Clustering": _CoClustering,
    }
    params = dict(_BEST_PARAMS[algo_name])  # copy
    params["rating_min"] = 1
    params["rating_max"] = R

    algo = cls_map[algo_name](**params)
    algo.fit(train_df, n_users, n_items)

    u_te = test_df["userId"].values - 1
    j_te = test_df["movieId"].values - 1
    y_pred = algo.predict(u_te, j_te).astype(float)
    y_true = test_df["rating"].values.astype(float)

    mae  = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    return mae, rmse


# ═══════════════════════════════════════════════════════════════════════════════
# 14. CHECKPOINT HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

def save_ckpt(obj, name: str):
    path = os.path.join(CHECKPOINT_DIR, f"{name}.pkl")
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"    ✓ checkpoint saved → {path}")


def load_ckpt(name: str):
    for base in [CHECKPOINT_DIR, PREV_CHECKPOINT_DIR]:
        path = os.path.join(base, f"{name}.pkl")
        if os.path.exists(path):
            with open(path, "rb") as f:
                return pickle.load(f)
    return None


def save_model(model: MMMF, name: str):
    path = os.path.join(CHECKPOINT_DIR, f"{name}.pt")
    torch.save(model.state_dict(), path)


def load_model(model: MMMF, name: str) -> bool:
    for base in [CHECKPOINT_DIR, PREV_CHECKPOINT_DIR]:
        path = os.path.join(base, f"{name}.pt")
        if os.path.exists(path):
            model.load_state_dict(torch.load(path, map_location=device))
            return True
    return False

# ═══════════════════════════════════════════════════════════════════════════════
# 15. ONE FULL ST-MMMF RUN  (Algorithm 1)
# ═══════════════════════════════════════════════════════════════════════════════

def run_one(dataset_name: str, train_df: pd.DataFrame, test_df: pd.DataFrame,
            lam: float, run_id: int):
    """
    Execute Algorithm 1 exactly for one random seed.
    Returns a dict with mae/rmse lists for all 5 algorithms across 50 iterations.
    """
    n_users = max(train_df["userId"].max(), test_df["userId"].max())
    n_items = max(train_df["movieId"].max(), test_df["movieId"].max())

    ck_key   = f"{dataset_name}_run{run_id}"
    ck_state = load_ckpt(ck_key + "_state")
    model    = MMMF(n_users, n_items, d=D, R=R).to(device)
    loaded   = load_model(model, ck_key + "_model")

    if ck_state and loaded:
        print(f"  [resume] run {run_id} from iter {ck_state['iter'] + 1}")
        results      = ck_state["results"]
        current_tr   = ck_state["current_train"]
        start_iter   = ck_state["iter"] + 1
    else:
        results = {
            "st_mmmf": {"mae": [], "rmse": []},
            "SVD":     {"mae": [], "rmse": []},
            "NMF":     {"mae": [], "rmse": []},
            "SVD++":   {"mae": [], "rmse": []},
            "Co-Clustering": {"mae": [], "rmse": []},
            "aug_stats": [],
        }
        current_tr   = train_df.copy()
        start_iter   = 0

    outer_bar = tqdm(range(start_iter, N_OUTER_ITERS),
                     desc=f"  ST-MMMF [{dataset_name}] run {run_id}",
                     ncols=100, unit="iter")

    for it in outer_bar:
        t0 = time.time()

        # ── Stage 1: train MMMF on current Y (Algorithm 1 line 3) ────────────
        train_mmmf(model, current_tr, lam, desc=f"iter {it+1:02d}")

        # ── Evaluate ST-MMMF ──────────────────────────────────────────────────
        mae_mmmf, rmse_mmmf = evaluate_mmmf(model, test_df)
        results["st_mmmf"]["mae"].append(mae_mmmf)
        results["st_mmmf"]["rmse"].append(rmse_mmmf)

        # ── Evaluate Surprise baselines on augmented training data ─────────────
        for bname in ["SVD", "NMF", "SVD++", "Co-Clustering"]:
            try:
                bm, br = run_baseline(bname, current_tr, test_df)
            except Exception:
                prev_mae  = results[bname]["mae"][-1]  if results[bname]["mae"]  else 1.0
                prev_rmse = results[bname]["rmse"][-1] if results[bname]["rmse"] else 1.0
                bm, br = prev_mae, prev_rmse
            results[bname]["mae"].append(bm)
            results[bname]["rmse"].append(br)

        # ── Build observed-entry mask ──────────────────────────────────────────
        Y_mask = torch.zeros(n_users, n_items, dtype=torch.bool, device=device)
        u_idx  = torch.tensor(current_tr["userId"].values - 1, dtype=torch.long)
        j_idx  = torch.tensor(current_tr["movieId"].values - 1, dtype=torch.long)
        Y_mask[u_idx, j_idx] = True

        # ── Stage 2: Augmentation (Equation 12) ───────────────────────────────
        cands   = augmentation_candidates(model, Y_mask, TAU1)
        sampled = balance_and_sample(cands, current_tr)

        # ── Stage 3: Refinement (Equation 13) ─────────────────────────────────
        remove  = refinement_candidates(model, Y_mask, TAU2)

        # ── Stage 4: Update Y ─────────────────────────────────────────────────
        # Remove low-confidence observed entries
        if remove:
            keep_mask = ~pd.Series(
                [(int(r.userId) - 1, int(r.movieId) - 1) in remove
                 for r in current_tr.itertuples(index=False)]
            ).values
            current_tr = current_tr[keep_mask].reset_index(drop=True)

        # Add high-confidence predictions
        if sampled:
            new_rows = pd.DataFrame({
                "userId":  [u + 1 for u, j, rating in sampled],
                "movieId": [j + 1 for u, j, rating in sampled],
                "rating":  [rating for u, j, rating in sampled],
            })
            current_tr = pd.concat([current_tr, new_rows], ignore_index=True)

        # ── Log augmentation stats (for Table 4 reproduction) ─────────────────
        results["aug_stats"].append({
            "iteration":    it + 1,
            "n_observed":   len(current_tr),
            "n_unobserved": n_users * n_items - len(current_tr),
            "n_high_conf":  len(cands),
            "n_augmented":  len(sampled),
            "n_refined":    len(remove),
        })

        elapsed = time.time() - t0
        outer_bar.set_postfix({
            "MAE":  f"{mae_mmmf:.4f}",
            "RMSE": f"{rmse_mmmf:.4f}",
            "+aug": len(sampled),
            "-ref": len(remove),
            "s":    f"{elapsed:.0f}",
        })

        # ── Checkpoint ────────────────────────────────────────────────────────
        save_ckpt({"iter": it, "results": results, "current_train": current_tr},
                  ck_key + "_state")
        save_model(model, ck_key + "_model")

    return results


# ═══════════════════════════════════════════════════════════════════════════════
# 16. AVERAGE OVER N_RUNS   (paper: "averaged over three runs")
# ═══════════════════════════════════════════════════════════════════════════════

def run_experiment(dataset_name: str, full_df: pd.DataFrame, lam: float):
    """Run N_RUNS independent trials and average results."""
    all_runs = []
    for run_id in range(N_RUNS):
        seed = 42 + run_id * 7           # different seed each run
        train_df, test_df = train_test_split_df(full_df, seed=seed)
        print(f"\n── Run {run_id + 1}/{N_RUNS}  "
              f"train={len(train_df)}, test={len(test_df)} ──")
        res = run_one(dataset_name, train_df, test_df, lam, run_id=run_id)
        all_runs.append(res)

    # Average metrics across runs
    averaged = {
        "st_mmmf": {"mae": [], "rmse": []},
        "SVD":     {"mae": [], "rmse": []},
        "NMF":     {"mae": [], "rmse": []},
        "SVD++":   {"mae": [], "rmse": []},
        "Co-Clustering": {"mae": [], "rmse": []},
        "aug_stats": all_runs[0]["aug_stats"],  # from first run
    }
    for metric in ["mae", "rmse"]:
        for algo_key in ["st_mmmf", "SVD", "NMF", "SVD++", "Co-Clustering"]:
            vals_per_iter = list(zip(*[r[algo_key][metric] for r in all_runs]))
            averaged[algo_key][metric] = [np.mean(v) for v in vals_per_iter]

    return averaged, all_runs


# ═══════════════════════════════════════════════════════════════════════════════
# 17. CONFUSION MATRICES FOR TABLES 2 & 3
# ═══════════════════════════════════════════════════════════════════════════════

def compute_confusion_tables(dataset_name: str, full_df: pd.DataFrame,
                             lam: float, seed: int = 42):
    """
    Reproduce Tables 2 & 3: confusion matrices and HR@K for
    training-set and test-set after iteration 1 and iteration 50.
    """
    print(f"\n[Tables 2/3] Computing confusion matrices for {dataset_name} ...")
    train_df, test_df = train_test_split_df(full_df, seed=seed)
    n_users = max(train_df["userId"].max(), test_df["userId"].max())
    n_items = max(train_df["movieId"].max(), test_df["movieId"].max())

    def _get_predictions(model, df):
        with torch.no_grad():
            Yhat = model.discretize().cpu().numpy()
        u = df["userId"].values - 1
        j = df["movieId"].values - 1
        return df["rating"].values, Yhat[u, j]

    # ── After 1st iteration ───────────────────────────────────────────────────
    model_1 = MMMF(n_users, n_items).to(device)
    train_mmmf(model_1, train_df, lam, desc="Tables 2/3 — iter 1")

    yt_tr, yp_tr = _get_predictions(model_1, train_df)
    yt_te, yp_te = _get_predictions(model_1, test_df)
    cm_tr_1, hr_tr_1 = confusion_and_hr(yt_tr, yp_tr)
    cm_te_1, hr_te_1 = confusion_and_hr(yt_te, yp_te)

    # ── After 50th iteration (use the checkpoint from run_0) ──────────────────
    ck_key  = f"{dataset_name}_run0"
    state   = load_ckpt(ck_key + "_state")
    model_L = MMMF(n_users, n_items).to(device)
    if not load_model(model_L, ck_key + "_model") or state is None:
        # No checkpoint: run a quick 50-iter pass
        print("  (No saved model found — re-running to get iter-50 model)")
        run_one(dataset_name, train_df, test_df, lam, run_id=99)
        load_model(model_L, f"{dataset_name}_run99_model")
        state = load_ckpt(f"{dataset_name}_run99_state")

    final_tr = state["current_train"] if state else train_df
    yt_tr_L, yp_tr_L = _get_predictions(model_L, final_tr)
    yt_te_L, yp_te_L = _get_predictions(model_L, test_df)
    cm_tr_L, hr_tr_L = confusion_and_hr(yt_tr_L, yp_tr_L)
    cm_te_L, hr_te_L = confusion_and_hr(yt_te_L, yp_te_L)

    return {
        "cm_train_iter1":  cm_tr_1,  "hr_train_iter1":  hr_tr_1,
        "cm_test_iter1":   cm_te_1,  "hr_test_iter1":   hr_te_1,
        "cm_train_iter50": cm_tr_L,  "hr_train_iter50": hr_tr_L,
        "cm_test_iter50":  cm_te_L,  "hr_test_iter50":  hr_te_L,
    }


# ═══════════════════════════════════════════════════════════════════════════════
# 18. VISUALISATION  (reproducing Figures 5 & 6 and Tables 2–4)
# ═══════════════════════════════════════════════════════════════════════════════

ALGO_STYLES = {
    "st_mmmf":       dict(marker="o", color="#1f77b4", label="ST-MMMF"),
    "NMF":           dict(marker="o", color="#ff7f0e", label="NMF"),
    "SVD":           dict(marker="+", color="#2ca02c", label="SVD"),
    "SVD++":         dict(marker="*", color="#d62728", label="SVD++"),
    "Co-Clustering": dict(marker="D", color="#9467bd", label="Co-Clustering"),
}


def plot_figure5(df_100k: pd.DataFrame, df_1m: pd.DataFrame):
    """Figure 5: Rating distribution bar charts."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    for ax, df, title in zip(axes, [df_100k, df_1m],
                             ["(a) MovieLens 100K", "(b) MovieLens 1M"]):
        cnt = df["rating"].value_counts().sort_index()
        ax.bar(cnt.index, cnt.values, color="#4c72b0", edgecolor="black", width=0.6)
        ax.set_xlabel("Rating")
        ax.set_ylabel("Count")
        ax.set_title(title)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, "fig5_rating_distribution.png")
    plt.savefig(p, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  Saved → {p}")


def plot_figure6(res_100k: dict, res_1m: dict):
    """
    Figure 6: MAE and RMSE across 50 iterations for all 5 algorithms.
    2 rows (MAE, RMSE) × 2 columns (100K, 1M).
    """
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    titles = [
        ["(a) MAE for MovieLens 100K",  "(b) MAE for MovieLens 1M"],
        ["(c) RMSE for MovieLens 100K", "(d) RMSE for MovieLens 1M"],
    ]
    for row, metric in enumerate(["mae", "rmse"]):
        for col, res in enumerate([res_100k, res_1m]):
            ax    = axes[row][col]
            iters = list(range(1, N_OUTER_ITERS + 1))
            for ak, sty in ALGO_STYLES.items():
                vals = res[ak][metric]
                if vals:
                    ax.plot(iters[:len(vals)], vals,
                            f"{sty['marker']}-", color=sty["color"],
                            label=sty["label"], markersize=4, linewidth=1.2)
            ax.set_xlabel("# Iteration")
            ax.set_ylabel("MAE" if metric == "mae" else "RMSE")
            ax.set_title(titles[row][col])
            ax.legend(fontsize=7)
            ax.grid(True, alpha=0.3)
    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, "fig6_performance_curves.png")
    plt.savefig(p, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  Saved → {p}")


def print_and_save_confusion_table(cm: np.ndarray, hr_k: np.ndarray,
                                   title: str, fname: str):
    """
    Print confusion matrix + HR@K in the same style as Tables 2 & 3,
    and save as a PNG image.
    """
    R = cm.shape[0]
    print(f"\n{'─'*70}")
    print(f"  {title}")
    print(f"{'─'*70}")
    header = "Actual \\ Pred |" + "".join(f"  {r+1:6d}  " for r in range(R))
    header += " | HR@0    HR@1    HR@2    HR@3    HR@4"
    print(header)
    for r in range(R):
        row = f"     r={r+1:1d}       |"
        row += "".join(f" {cm[r, c]:7d} " for c in range(R))
        row += " | " + "  ".join(f"{hr_k[r, k]:.4f}" for k in range(R))
        print(row)

    # Save as PNG
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(title, fontsize=10, fontweight="bold")

    # Confusion heatmap
    im = axes[0].imshow(cm, cmap="Blues")
    axes[0].set_xticks(range(R)); axes[0].set_yticks(range(R))
    axes[0].set_xticklabels([str(i + 1) for i in range(R)])
    axes[0].set_yticklabels([str(i + 1) for i in range(R)])
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
    axes[0].set_title("Confusion Matrix")
    for i in range(R):
        for j in range(R):
            clr = "white" if cm[i, j] > cm.max() * 0.55 else "black"
            axes[0].text(j, i, str(cm[i, j]),
                         ha="center", va="center", fontsize=7, color=clr)
    plt.colorbar(im, ax=axes[0])

    # HR@K bar chart
    x = np.arange(R); w = 0.15
    for k in range(min(5, R)):
        axes[1].bar(x + k * w, hr_k[:, k], w, label=f"HR@{k}")
    axes[1].set_xticks(x + w * 2)
    axes[1].set_xticklabels([f"r={i+1}" for i in range(R)])
    axes[1].set_xlabel("Actual Rating"); axes[1].set_ylabel("Fraction")
    axes[1].set_title("HR@K per Rating Label")
    axes[1].legend(fontsize=7)

    plt.tight_layout()
    p = os.path.join(OUTPUT_DIR, fname)
    plt.savefig(p, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  Saved → {p}")


def print_table4(aug_stats: list):
    """Table 4: Effect of data augmentation (first 5 iterations)."""
    print("\n  Table 4: Effect of data augmentation (first 5 iterations)")
    print(f"  {'Iter':>4} | {'# Observed':>12} | {'# Unobserved':>13} | "
          f"{'# High-conf':>12} | {'# Augmented':>12} | {'# Overlap':>10}")
    print("  " + "─" * 75)
    prev_hc = None
    for s in aug_stats[:5]:
        overlap = "N/A"
        if prev_hc is not None:
            # paper: "overlap = high-conf entries that remain high-conf"
            # approximated as min(prev_hc, current_hc) for display
            overlap = str(min(prev_hc, s["n_high_conf"]))
        print(f"  {s['iteration']:>4} | {s['n_observed']:>12,} | "
              f"{s['n_unobserved']:>13,} | {s['n_high_conf']:>12,} | "
              f"{s['n_augmented']:>12,} | {overlap:>10}")
        prev_hc = s["n_high_conf"]


def save_csv_results(res_100k: dict, res_1m: dict):
    rows = []
    for dname, res in [("100K", res_100k), ("1M", res_1m)]:
        n = len(res["st_mmmf"]["mae"])
        for i in range(n):
            row = {"Dataset": dname, "Iteration": i + 1}
            for ak in ["st_mmmf", "SVD", "NMF", "SVD++", "Co-Clustering"]:
                algo_label = "ST-MMMF" if ak == "st_mmmf" else ak
                row[f"{algo_label}_MAE"]  = res[ak]["mae"][i]
                row[f"{algo_label}_RMSE"] = res[ak]["rmse"][i]
            rows.append(row)
    df = pd.DataFrame(rows)
    p  = os.path.join(OUTPUT_DIR, "performance_results.csv")
    df.to_csv(p, index=False)
    print(f"\n  Saved numerical results → {p}")
    print("\n  ── Last-iteration results ──")
    last = df.groupby("Dataset").last().reset_index()
    print(last.to_string(index=False))


# ═══════════════════════════════════════════════════════════════════════════════
# 19. MAIN
# ═══════════════════════════════════════════════════════════════════════════════

def main():
    t_total = time.time()

    # ── Load & preprocess datasets ────────────────────────────────────────────
    print("\n[1/6] Loading datasets ...")
    df_100k = load_100k()
    df_1m   = load_1m()

    # ── Figure 5: rating distributions ───────────────────────────────────────
    print("\n[2/6] Plotting Figure 5 (rating distributions) ...")
    plot_figure5(df_100k, df_1m)

    # ── Select λ for each dataset via hold-out validation ────────────────────
    print("\n[3/6] λ selection (paper §5.1) ...")
    # Use a fixed seed=42 split for λ search (we'll use N_RUNS different seeds
    # for the actual experiment)
    train_100k_init, _ = train_test_split_df(df_100k, seed=42)
    train_1m_init,   _ = train_test_split_df(df_1m,   seed=42)

    n_u_100k = df_100k["userId"].nunique();  n_i_100k = df_100k["movieId"].nunique()
    n_u_1m   = df_1m["userId"].nunique();    n_i_1m   = df_1m["movieId"].nunique()

    print("  100K:")
    lam_100k = select_lambda(train_100k_init, n_u_100k, n_i_100k)
    print("  1M:")
    lam_1m   = select_lambda(train_1m_init,   n_u_1m,   n_i_1m)

    # ── Tune Surprise baselines ────────────────────────────────────────────────
    print("\n[4/6] Tuning Surprise baselines ...")
    tune_baselines(train_100k_init)   # params shared across both datasets

    # ── Run ST-MMMF experiment (3 runs each) ──────────────────────────────────
    print("\n[5/6] Running ST-MMMF — MovieLens 100K ...")
    res_100k, runs_100k = run_experiment("100K", df_100k, lam_100k)
    save_ckpt(res_100k, "final_results_100k")

    print("\n[5/6] Running ST-MMMF — MovieLens 1M ...")
    res_1m,   runs_1m   = run_experiment("1M",   df_1m,   lam_1m)
    save_ckpt(res_1m, "final_results_1m")

    # ── Figure 6: performance curves ──────────────────────────────────────────
    print("\n[6/6] Generating figures & tables ...")
    plot_figure6(res_100k, res_1m)

    # ── Tables 2 & 3: confusion matrices ──────────────────────────────────────
    for dname, df, lam in [("100K", df_100k, lam_100k), ("1M", df_1m, lam_1m)]:
        conf = compute_confusion_tables(dname, df, lam)
        for split in ["train", "test"]:
            for it_label, it_key in [("1st", "iter1"), ("50th", "iter50")]:
                cm   = conf[f"cm_{split}_{it_key}"]
                hr_k = conf[f"hr_{split}_{it_key}"]
                title = (f"MovieLens {dname} — {split}-set — "
                         f"Confusion after {it_label} iteration")
                fname = f"table_{dname}_{split}_{it_key}.png"
                print_and_save_confusion_table(cm, hr_k, title, fname)

    # ── Table 4: augmentation statistics ──────────────────────────────────────
    print("\n  === MovieLens 100K ===")
    print_table4(res_100k["aug_stats"])
    print("\n  === MovieLens 1M ===")
    print_table4(res_1m["aug_stats"])

    # ── Save all numeric results to CSV ───────────────────────────────────────
    save_csv_results(res_100k, res_1m)

    print(f"\n{'='*65}")
    print(f"  DONE.  Total time: {(time.time()-t_total)/60:.1f} min")
    print(f"  All outputs in: {OUTPUT_DIR}")
    print(f"{'='*65}")


# ─── entry point ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()


main()


